# 05 — Community Detection on the Semantic Graph

In notebook 04 I built a kNN graph and explored connected components. The conclusion was that connected components are too coarse — at threshold 0.72, nearly half the complaints were isolated nodes, and the large component that remained was more of a broad semantic territory than a meaningful community.

Two things change in this notebook:

1. **No similarity threshold.** Pure kNN — every complaint gets exactly k neighbours, full coverage. The cosine similarity survives as an edge weight so Louvain still has quality signal, it just doesn't use it as a hard gate.
2. **k=10 instead of k=5.** Gives Louvain more local signal to work with. k=5 is noted as a comparison point.

The question: does Louvain find meaningful complaint communities inside the semantic graph, or is it just partitioning noise?


## Setup


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import numpy as np
import pandas as pd
import networkx as nx
import community as community_louvain  # pip install python-louvain
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

from src.embeddings import load_model, generate_embeddings
from src.retrieval import get_neighbors


## Load data

Same sample as before — keeping the dataset fixed makes results comparable across notebooks.


In [ ]:
df_sample = pd.read_parquet("../data/processed/complaints_50k.parquet")

texts = (
    df_sample["Consumer complaint narrative"]
    .dropna()
    .astype(str)
    .tolist()
)

sample_texts = texts[:2000]

print(f"{len(sample_texts):,} complaints loaded.")


## Generate embeddings


In [ ]:
model = load_model()

embeddings = generate_embeddings(sample_texts, model)

embeddings.shape


## UMAP projection

Same parameters as notebook 04 so the 2D layout is directly comparable.


In [ ]:
import umap.umap_ as umap

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42
)

embedding_2d = reducer.fit_transform(embeddings)

print(f"2D projection shape: {embedding_2d.shape}")


## Build the kNN graph — no threshold this time

In notebook 04 I used k=5 with a 0.72 similarity threshold. That left 925 of 2,000 complaints as isolated nodes — nearly half the dataset invisible to any graph algorithm before it even started.

Here I drop the threshold entirely. Every complaint gets exactly k neighbours. The similarity score is kept as an edge weight so the graph still encodes how strong each connection is, but nothing gets gated out.

k=10 gives Louvain more local structure to work with. The cell below also shows k=5 stats so the trade-off is visible.


In [ ]:
def build_knn_graph(embeddings, k):
    """
    Pure kNN graph, no threshold.
    Each node gets exactly k outgoing edges (its k nearest neighbours).
    Adding edges to an nx.Graph() automatically handles symmetry —
    if A→B and B→A both exist, it's still one undirected edge.
    The weight is set to the mean similarity when both directions are added.
    """
    G = nx.Graph()
    G.add_nodes_from(range(len(embeddings)))

    for i in range(len(embeddings)):
        neighbors, similarities = get_neighbors(
            embeddings,
            query_index=i,
            k=k + 1  # +1 because the query itself comes back
        )
        for neighbor in neighbors:
            if neighbor == i:
                continue
            sim = similarities[neighbor]
            if G.has_edge(i, neighbor):
                # edge already exists from the other direction — average the weights
                existing = G[i][neighbor]["weight"]
                G[i][neighbor]["weight"] = (existing + sim) / 2
            else:
                G.add_edge(i, neighbor, weight=sim)

    return G


for k in [5, 10]:
    G_tmp = build_knn_graph(embeddings, k)
    degrees = [d for _, d in G_tmp.degree()]
    print(f"k={k}  nodes: {G_tmp.number_of_nodes():,}  edges: {G_tmp.number_of_edges():,}  "
          f"degree mean: {np.mean(degrees):.1f}  max: {max(degrees)}")


In [ ]:
# Build the graph we'll actually use
K = 10
G = build_knn_graph(embeddings, K)

print(f"Graph built with k={K}")
print(f"  Nodes: {G.number_of_nodes():,}")
print(f"  Edges: {G.number_of_edges():,}")
print(f"  Isolated nodes: {sum(1 for _, d in G.degree() if d == 0)}")


## Quick look at the degree distribution

Worth checking before running Louvain. If a handful of nodes have extremely high degree — say 5x the mean — those are hub nodes: very generic complaints that land near everything in embedding space. They'll pull communities toward themselves and distort the partition.

With a pure kNN graph and k=10, the expected degree is around 10–20 (10 outgoing + some incoming reciprocals). Anything much higher is a flag.


In [ ]:
degrees = [d for _, d in G.degree()]

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=degrees,
    nbinsx=40,
    marker_color="#6366f1",
    opacity=0.8,
    name="degree"
))
fig.add_vline(
    x=np.mean(degrees),
    line_dash="dash",
    line_color="#ef4444",
    annotation_text=f"mean = {np.mean(degrees):.1f}",
    annotation_position="top right"
)
fig.update_layout(
    title="Node degree distribution (k=10, no threshold)",
    xaxis_title="Degree",
    yaxis_title="Count",
    showlegend=False,
    width=750,
    height=380
)
fig.show()

# Flag any suspicious hubs
hub_threshold = np.mean(degrees) + 3 * np.std(degrees)
hubs = [(n, d) for n, d in G.degree() if d > hub_threshold]
print(f"Nodes with degree > mean + 3σ ({hub_threshold:.0f}): {len(hubs)}")
if hubs:
    top = sorted(hubs, key=lambda x: -x[1])[:3]
    for node, deg in top:
        print(f"  Node {node}, degree {deg}: {sample_texts[node][:150]}")


## Run Louvain

A few things worth knowing going in:

- **Louvain is non-deterministic.** It uses random tie-breaking internally. We fix a seed for reproducibility and check stability across seeds below.
- **Resolution controls granularity.** Higher → more, smaller communities. Default is 1.0. We'll sweep it.
- **Edge weights matter.** We pass `weight="weight"` so Louvain uses cosine similarity rather than treating all edges as equal.


In [ ]:
SEED = 42

partition = community_louvain.best_partition(
    G,
    weight="weight",
    resolution=1.0,
    random_state=SEED
)

community_ids = np.array([partition[i] for i in range(len(partition))])
n_communities = len(set(community_ids))
modularity = community_louvain.modularity(partition, G, weight="weight")

print(f"Communities found: {n_communities}")
print(f"Modularity:        {modularity:.4f}")
print()

sizes = sorted(Counter(community_ids).values(), reverse=True)
print(f"Community sizes: {sizes}")
print(f"Largest:  {sizes[0]} ({sizes[0]/len(sample_texts)*100:.1f}% of sample)")
print(f"Smallest: {sizes[-1]}")
print(f"Median:   {int(np.median(sizes))}")


## Stability check

Before spending time interpreting communities, it's worth asking: are these stable across runs, or is Louvain landing on different partitions each time?

We run it 10 times with different seeds and check two things:
- whether the number of communities is consistent
- co-assignment consistency: for pairs of nodes in the same community on run 0, how often are they also together on other runs?


In [ ]:
N_RUNS = 10
partitions = []

for seed in range(N_RUNS):
    p = community_louvain.best_partition(
        G, weight="weight", resolution=1.0, random_state=seed
    )
    partitions.append(np.array([p[i] for i in range(len(p))]))

n_comms_per_run = [len(set(p)) for p in partitions]
print(f"Community counts across {N_RUNS} runs: {n_comms_per_run}")
print(f"Range: {min(n_comms_per_run)}–{max(n_comms_per_run)}")

# Co-assignment consistency
import random
random.seed(42)
ref = partitions[0]
all_pairs = [(i, j) for i in range(0, 500) for j in range(i+1, 500)]
co_pairs = [(a, b) for a, b in all_pairs if ref[a] == ref[b]]
co_pairs_sample = random.sample(co_pairs, min(3000, len(co_pairs)))

scores = []
for p in partitions[1:]:
    agree = sum(1 for a, b in co_pairs_sample if p[a] == p[b])
    scores.append(agree / len(co_pairs_sample))

print(f"\nCo-assignment consistency (pairs co-assigned in run 0):")
print(f"  Mean: {np.mean(scores):.3f}  Std: {np.std(scores):.3f}")
print(f"  Values: {[round(s, 3) for s in scores]}")
print("\n  >0.8 solid  |  0.6–0.8 fuzzy boundaries  |  <0.6 worry")


## Resolution sweep

Louvain's default resolution can miss communities that are either too small or too large relative to the overall graph size. Worth sweeping to see if there's a natural scale where the partition feels right.


In [ ]:
resolutions = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0]
res_results = []

for res in resolutions:
    p = community_louvain.best_partition(
        G, weight="weight", resolution=res, random_state=SEED
    )
    mod = community_louvain.modularity(p, G, weight="weight")
    comm_sizes = sorted(Counter(p.values()).values(), reverse=True)
    res_results.append({
        "resolution": res,
        "n_communities": len(comm_sizes),
        "modularity": round(mod, 4),
        "largest": comm_sizes[0],
        "median_size": int(np.median(comm_sizes)),
        "singletons": sum(1 for s in comm_sizes if s == 1),
    })

res_df = pd.DataFrame(res_results)
print(res_df.to_string(index=False))


In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=res_df["resolution"], y=res_df["n_communities"],
    mode="lines+markers", name="n communities",
    line=dict(color="#6366f1")
))

fig.update_layout(
    title="Number of communities vs resolution",
    xaxis_title="Resolution",
    yaxis_title="Communities",
    width=700, height=350
)
fig.show()


## Visualise communities on the UMAP projection

The moment of truth. If Louvain found real semantic structure, communities should form spatially coherent regions in UMAP space — because the 2D layout was built from the same embeddings.

Scattered communities across the UMAP are a red flag. Coherent blobs are a good sign.


In [ ]:
plot_df = pd.DataFrame({
    "x": embedding_2d[:, 0],
    "y": embedding_2d[:, 1],
    "community": [f"C{c}" for c in community_ids],
    "index": range(len(sample_texts)),
    "text_preview": [t[:120] for t in sample_texts]
})

fig = px.scatter(
    plot_df,
    x="x", y="y",
    color="community",
    hover_data={"index": True, "text_preview": True, "x": False, "y": False},
    opacity=0.65,
    size_max=6,
    title=f"Louvain communities on UMAP (k={K}, resolution=1.0, {n_communities} communities)"
)

fig.update_traces(marker=dict(size=4))
fig.update_layout(width=950, height=750)
fig.show()


## Interpreting communities

The algorithm assigned IDs, not labels. Now we do the interpretive work. Three lenses:

1. **Product category distribution** — do communities map to financial product types?
2. **Most central complaint per community** — the node with highest within-community degree; read these.
3. **TF-IDF keywords** — what vocabulary is distinctive to each community?


In [ ]:
# 1. Product distribution per community
product_col = "Product"  # adjust to your actual CFPB column name if different

if product_col in df_sample.columns:
    df_plot = df_sample.iloc[:2000].copy()
    df_plot["community"] = community_ids

    prod_dist = (
        df_plot.groupby(["community", product_col])
        .size()
        .reset_index(name="count")
    )

    print("Top products per community:\n")
    for comm_id in sorted(set(community_ids)):
        sub = prod_dist[prod_dist["community"] == comm_id].sort_values("count", ascending=False)
        total = sub["count"].sum()
        print(f"C{comm_id} (n={total}):")
        for _, row in sub.head(3).iterrows():
            pct = 100 * row["count"] / total
            print(f"  {str(row[product_col])[:55]:<55} {row['count']:>4}  ({pct:.0f}%)")
        print()
else:
    print(f"Column '{product_col}' not found. Available: {list(df_sample.columns[:10])}")


In [ ]:
# 2. Most central complaint per community
print("Most central complaint per community\n" + "="*65)

for comm_id in sorted(set(community_ids)):
    nodes_in_comm = [n for n, c in partition.items() if c == comm_id]
    subG = G.subgraph(nodes_in_comm)
    central_node = max(subG.degree(), key=lambda x: x[1])[0]
    size = len(nodes_in_comm)
    print(f"\nC{comm_id} ({size} complaints) — node {central_node}")
    print(sample_texts[central_node][:350])
    print("-"*65)


In [ ]:
# 3. TF-IDF keywords per community
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings("ignore")

community_docs = {
    comm_id: " ".join(sample_texts[i] for i in range(len(sample_texts))
                      if community_ids[i] == comm_id)
    for comm_id in sorted(set(community_ids))
}

comm_ids_ordered = sorted(community_docs.keys())
corpus = [community_docs[c] for c in comm_ids_ordered]

vec = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1, 2),
    sublinear_tf=True
)
tfidf = vec.fit_transform(corpus)
feature_names = vec.get_feature_names_out()

print("Top TF-IDF keywords per community\n")
for i, comm_id in enumerate(comm_ids_ordered):
    row = tfidf[i].toarray().flatten()
    top_idx = row.argsort()[::-1][:12]
    keywords = ", ".join(feature_names[j] for j in top_idx)
    size = (community_ids == comm_id).sum()
    print(f"C{comm_id} (n={size:>4}): {keywords}")


## Coherence check: are communities semantically tighter than random?

Louvain finds communities in the graph topology. That doesn't automatically mean they're semantically coherent — it means they're well-connected. We need to verify that complaints *within* a community are actually more similar to each other in embedding space than to complaints outside it.

If intra-community similarity ≈ inter-community similarity, the communities are graph artefacts, not semantic ones.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# Full similarity matrix — fine for 2k samples (~30MB)
sim_matrix = cos_sim(embeddings)

# Sample-based: 300 nodes, all pairs
np.random.seed(42)
sample_idx = np.random.choice(len(sample_texts), size=300, replace=False)
sample_comms = community_ids[sample_idx]
sample_sims = sim_matrix[np.ix_(sample_idx, sample_idx)]

intra, inter = [], []
for i in range(len(sample_idx)):
    for j in range(i + 1, len(sample_idx)):
        s = sample_sims[i, j]
        if sample_comms[i] == sample_comms[j]:
            intra.append(s)
        else:
            inter.append(s)

print(f"Intra-community similarity — mean: {np.mean(intra):.4f}  std: {np.std(intra):.4f}")
print(f"Inter-community similarity — mean: {np.mean(inter):.4f}  std: {np.std(inter):.4f}")
print(f"Ratio (intra/inter):              {np.mean(intra)/np.mean(inter):.3f}")
print()
print(">1.10  communities are detecting real semantic groupings")
print("~1.00  communities are graph-topological artefacts")


In [ ]:
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=intra, nbinsx=50, opacity=0.65,
    marker_color="#6366f1", name=f"Intra (mean={np.mean(intra):.3f})",
    histnorm="probability density"
))
fig.add_trace(go.Histogram(
    x=inter, nbinsx=50, opacity=0.65,
    marker_color="#ef4444", name=f"Inter (mean={np.mean(inter):.3f})",
    histnorm="probability density"
))
fig.add_vline(x=np.mean(intra), line_dash="dash", line_color="#4f46e5")
fig.add_vline(x=np.mean(inter), line_dash="dash", line_color="#dc2626")
fig.update_layout(
    barmode="overlay",
    title="Intra vs inter-community cosine similarity",
    xaxis_title="Cosine similarity",
    yaxis_title="Density",
    width=800, height=400
)
fig.show()


## What did we find, and what comes next?

A few things to record once the cells above have run:

**If communities are spatially coherent in UMAP and intra/inter ratio > 1.1:** the graph has real semantic structure. Louvain is doing useful work. The product distribution check will tell you whether communities map onto CFPB product categories or onto something finer-grained (complaint situation types, institutions, specific issues).

**If intra/inter ratio is close to 1.0:** the graph topology and the embedding geometry are somewhat decoupled. This can happen when k=10 adds edges that are semantically weak — the kNN graph connects things that are "close enough" but not genuinely similar. Options: raise the threshold back (but not to 0.72 — try 0.60), or try HDBSCAN directly on embeddings as an alternative path.

**What should move to `src/` after this notebook:**
- `build_knn_graph(embeddings, k)` — used here and will be needed every time
- Nothing else yet; the Louvain wrapper and coherence check are still exploratory

**Natural next notebooks:**
- **06: Community labelling** — pipe TF-IDF keywords + representative complaints into an LLM call, get human-readable labels
- **06alt: HDBSCAN comparison** — cluster directly on embeddings, no graph, compare structure
- **07: Temporal drift** — do the same communities appear in 2019 vs 2022?
